# Notebook 03 — Preprocessing

**Purpose:** Fix every issue found in Notebook 02 and save a clean dataset.

**Input:**  `data/raw/updated_data.csv`  (never modified)  
**Output:** `data/processed/schemes_clean.csv`  (our working dataset from here on)

---

## The Cleaning Pipeline — 7 Steps

```
Raw CSV
  │
  ├─ Step 1: Drop the unnamed empty column
  ├─ Step 2: Drop exact duplicate rows
  ├─ Step 3: Fill missing values with safe defaults
  ├─ Step 4: Clean scheme_name (strip outer quotes + whitespace)
  ├─ Step 5: Clean all text fields (BOM, HTML entities, whitespace)
  ├─ Step 6: Standardize schemeCategory and level
  └─ Step 7: Save to data/processed/schemes_clean.csv
```

Every step prints a before/after count so you can see exactly what changed.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import re
import html
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 200)
print('Libraries loaded.')

In [ ]:
# ── Load raw data ─────────────────────────────────────────────────────────
RAW_PATH       = '../data/raw/updated_data.csv'
PROCESSED_PATH = '../data/processed/schemes_clean.csv'

df = pd.read_csv(RAW_PATH)
print(f'Raw data loaded: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')

---
## Step 1 — Drop the Unnamed Empty Column

**Problem:** There is a column with an empty string name `''` positioned between `schemeCategory` and `tags`. It was created by an extra comma in the CSV header during the Kaggle scrape. It contains no data.

**Fix:** Drop any column whose name is empty or named `'Unnamed: X'`.

**Python concept:** `df.drop(columns=[...])` removes the listed columns from the DataFrame.

In [ ]:
# Find columns with blank or 'Unnamed' names
cols_to_drop = [
    col for col in df.columns
    if str(col).strip() == '' or str(col).startswith('Unnamed:')
]

print(f'Columns to drop: {cols_to_drop}')

df = df.drop(columns=cols_to_drop)

print(f'Shape after Step 1: {df.shape}')
print(f'Remaining columns:   {list(df.columns)}')

---
## Step 2 — Drop Duplicate Rows

**Problem:** Some schemes appear more than once (exact copy or same slug).  
Having duplicates would make the recommendation engine suggest the same scheme multiple times.

**Fix:** Keep the first occurrence. Drop all subsequent exact duplicates.  
Then drop any row with a duplicate `slug` (same URL = same scheme).

**Python concept:** `df.drop_duplicates()` removes rows that are identical to a previous row.

In [ ]:
before = len(df)

# First pass: exact full-row duplicates
df = df.drop_duplicates(keep='first')
after_full = len(df)
print(f'Exact full-row duplicates removed: {before - after_full}')

# Second pass: duplicate slugs (same scheme, possibly slightly different scraped text)
df = df.drop_duplicates(subset=['slug'], keep='first')
after_slug = len(df)
print(f'Duplicate-slug rows removed:       {after_full - after_slug}')

print(f'Shape after Step 2: {df.shape}  ({before - after_slug} total rows removed)')

---
## Step 3 — Fill Missing Values

**Problem:** Several text fields have NaN (empty) cells.

**Strategy:**  
- Text fields used in NLP → fill with `''` (empty string). The NLP engine simply gets no signal from that field — it is not penalized.
- Category fields used in filtering → fill with a meaningful placeholder (`'Uncategorized'`, `'Unknown'`).
- `slug` → if missing, derive from `scheme_name` by lowercasing and replacing spaces with hyphens.

**Python concept:** `df['col'].fillna(value)` replaces NaN with the given value.

In [ ]:
print('Missing values BEFORE Step 3:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print()

In [ ]:
# Text fields → empty string
text_cols = ['details', 'benefits', 'eligibility', 'application', 'documents', 'tags']
for col in text_cols:
    df[col] = df[col].fillna('')

# Category fields → labeled placeholder
df['schemeCategory'] = df['schemeCategory'].fillna('Uncategorized')
df['level']          = df['level'].fillna('Unknown')

# slug → derive from scheme_name if missing
def make_slug(name):
    """Convert a scheme name into a URL-friendly slug."""
    s = str(name).lower().strip()
    s = re.sub(r'[^a-z0-9\s-]', '', s)   # keep only letters, digits, spaces, hyphens
    s = re.sub(r'\s+', '-', s)            # spaces → hyphens
    s = re.sub(r'-+', '-', s)             # collapse multiple hyphens
    return s.strip('-')

df['slug'] = df['slug'].fillna(df['scheme_name'].apply(make_slug))

print('Missing values AFTER Step 3:')
remaining = df.isnull().sum()
if remaining.sum() == 0:
    print('  None — all missing values resolved ✓')
else:
    print(remaining[remaining > 0])

---
## Step 4 — Clean `scheme_name`

**Problem:** Some names are wrapped in extra quote characters from the CSV scrape.  
Example: `'"Immediate Relief Assistance" under ...'`

**Fix:** Strip leading/trailing whitespace and outer quote characters.

**Python concept:** `str.strip(chars)` removes the listed characters from both ends of a string.

In [ ]:
before_sample = df['scheme_name'].head(3).to_list()

df['scheme_name'] = (
    df['scheme_name']
    .str.strip()          # remove leading/trailing whitespace
    .str.strip('"')       # remove outer double-quote characters
    .str.strip("'")       # remove outer single-quote characters
    .str.strip()          # strip whitespace again after removing quotes
)

after_sample = df['scheme_name'].head(3).to_list()

print('scheme_name before cleaning:')
for s in before_sample:
    print(f'  {repr(s[:80])}')

print()
print('scheme_name after cleaning:')
for s in after_sample:
    print(f'  {repr(s[:80])}')

---
## Step 5 — Clean All Text Fields

**Problems in text fields:**
1. BOM characters (`\ufeff`, ``) — invisible garbage from web scraping
2. HTML entities (`&amp;` → `&`, `&nbsp;` → space, `&lt;` → `<`)
3. Excessive newlines (`\n\n\n` → single space)
4. Multiple spaces → single space
5. Leading/trailing whitespace

**Python concepts:**
- `html.unescape(text)` — converts HTML entities to their actual characters
- `re.sub(pattern, replacement, text)` — replaces all regex matches in a string
- `str.strip()` — removes leading/trailing whitespace

**Why this matters for TF-IDF:**  
Without cleaning, `"government"` and `"government\n"` are two different tokens. The TF-IDF model treats them as completely different words, diluting the importance of both.

In [ ]:
def clean_text(text):
    """
    Clean a single text string:
    1. Return empty string if not a real string
    2. Remove BOM (Byte Order Mark) characters
    3. Decode HTML entities (&amp; → &, &nbsp; → space, etc.)
    4. Replace newlines and tab characters with a space
    5. Collapse multiple spaces into one
    6. Strip leading/trailing whitespace
    """
    if not isinstance(text, str):
        return ''
    
    # 1. Remove BOM and other zero-width characters
    text = text.replace('\ufeff', '').replace('\u200b', '').replace('\u00a0', ' ')
    
    # 2. Decode HTML entities
    text = html.unescape(text)
    
    # 3. Replace newlines and tabs with a space
    text = re.sub(r'[\n\r\t]+', ' ', text)
    
    # 4. Collapse multiple spaces into one
    text = re.sub(r' {2,}', ' ', text)
    
    # 5. Strip leading/trailing whitespace
    text = text.strip()
    
    return text


# Apply to all text columns
all_text_cols = ['scheme_name', 'details', 'benefits', 'eligibility',
                 'application', 'documents', 'tags']

for col in all_text_cols:
    df[col] = df[col].apply(clean_text)

print('Text cleaning applied to all columns.')

# Verify: show before/after for row 0 details field
print()
print('Sample cleaned `details` (first 300 chars of row 0):')
print(df['details'].iloc[0][:300])

---
## Step 6 — Standardize `schemeCategory` and `level`

**Problem:** Categories may have inconsistent formatting — extra spaces, wrong casing, or minor spelling variants.

**Fix:** Strip whitespace and apply consistent title-casing.  
This ensures `'education & learning'` and `'Education & Learning'` are treated as the same category.

**Python concept:** `str.title()` capitalizes the first letter of every word.

In [ ]:
# Standardize schemeCategory
df['schemeCategory'] = (
    df['schemeCategory']
    .str.strip()
    .str.title()   # "education & learning" → "Education & Learning"
)

# Standardize level — enforce known values only
valid_levels = {'Central', 'State', 'District', 'Unknown'}
df['level'] = df['level'].str.strip().str.title()
# Map anything unexpected to 'Unknown'
df['level'] = df['level'].apply(
    lambda x: x if x in valid_levels else 'Unknown'
)

print('Category standardization complete.')
print()
print('Level distribution after standardization:')
print(df['level'].value_counts())
print()
print('Top 15 categories after standardization:')
print(df['schemeCategory'].value_counts().head(15))

---
## Step 7 — Build `combined_text` column

**What is this?**  
For the TF-IDF engine in Phase 6, each scheme needs to be represented as a **single string** that captures all its important information. We concatenate the most content-rich fields into one combined field.

**Why not keep them separate?**  
TF-IDF vectorizes one document per scheme. Combining fields means one scheme → one vector → one similarity score. It's simpler and works well for our use case.

**Formula:**
```
combined_text = scheme_name + " " + schemeCategory + " " + details + " " + benefits + " " + eligibility + " " + tags
```

In [ ]:
df['combined_text'] = (
    df['scheme_name']    + ' ' +
    df['schemeCategory'] + ' ' +
    df['details']        + ' ' +
    df['benefits']       + ' ' +
    df['eligibility']    + ' ' +
    df['tags']
)

# Clean up any double spaces that may have crept in
df['combined_text'] = df['combined_text'].str.strip()
df['combined_text'] = df['combined_text'].apply(
    lambda t: re.sub(r' {2,}', ' ', t)
)

print('combined_text column created.')
print(f'Average length: {df["combined_text"].str.len().mean():.0f} characters')
print(f'Min length:     {df["combined_text"].str.len().min()} characters')
print(f'Max length:     {df["combined_text"].str.len().max()} characters')
print()
print('Sample combined_text (row 0, first 400 chars):')
print(df['combined_text'].iloc[0][:400])

---
## Final Validation — Check the Clean Dataset

In [ ]:
print('=' * 55)
print('FINAL CLEAN DATASET SUMMARY')
print('=' * 55)
print(f'Total rows:    {len(df)}')
print(f'Total columns: {len(df.columns)}')
print(f'Columns:       {list(df.columns)}')
print()
print('Null counts after all cleaning steps:')
nulls = df.isnull().sum()
if nulls.sum() == 0:
    print('  → Zero nulls in any column ✓')
else:
    print(nulls[nulls > 0])
print()
print('Remaining duplicates on slug:')
slug_dupes = df.duplicated(subset=['slug']).sum()
print(f'  → {slug_dupes} (should be 0)')

---
## Save the Cleaned Dataset

**`index=False`** — we don't want pandas to save the row numbers (0, 1, 2...) as a column in the CSV. That's internal index data that has no meaning.

In [ ]:
os.makedirs('../data/processed', exist_ok=True)

df.to_csv(PROCESSED_PATH, index=False, encoding='utf-8')

file_size_kb = os.path.getsize(PROCESSED_PATH) / 1024
print(f'Clean dataset saved to: {PROCESSED_PATH}')
print(f'File size: {file_size_kb:.1f} KB')
print()
print('Reload check — read back and confirm shape:')
verify = pd.read_csv(PROCESSED_PATH)
print(f'  Reloaded shape: {verify.shape}  ✓')

---
## Phase 2 Complete ✓

| Deliverable | Status |
|---|---|
| Raw data understood | ✓ Notebook 01 |
| All quality issues diagnosed | ✓ Notebook 02 |
| All issues fixed | ✓ This notebook |
| Clean CSV saved | ✓ `data/processed/schemes_clean.csv` |
| `combined_text` column built | ✓ Ready for TF-IDF in Phase 6 |

**Next phase:** Notebook 04 — Exploratory Data Analysis (EDA)  
We will use charts and statistics to understand patterns in the data before building any models.